# Módulo 3 · Clase 5 — Proyecto: automatización con Bash

**Curso de Inteligencia Artificial — Nivel básico**

---

## Objetivo del proyecto

En este proyecto construimos **un sistema de automatización real** aplicando todo lo del Módulo 3. El sistema gestiona experimentos de Machine Learning: organiza datos, lanza entrenamientos, analiza logs y hace backups automáticos.

```
SISTEMA DE AUTOMATIZACIÓN ML

setup.sh           → prepara el entorno del proyecto
lanzar.sh          → lanza un experimento (config + simulación)
analizar.sh        → analiza logs de entrenamiento
backup.sh          → backup con rotación automática
estado.sh          → informe completo del proyecto
```

Cada script usa las herramientas de las cuatro clases anteriores:

| Script | Habilidades usadas |
|--------|-------------------|
| `setup.sh` | `mkdir -p`, `chmod`, permisos, condicionales |
| `lanzar.sh` | Argumentos, variables, bucles, heredoc |
| `analizar.sh` | `grep`, `find`, pipes, redirecciones |
| `backup.sh` | `cp`/`mv`, `find -mtime`, funciones, rotación |
| `estado.sh` | Todo combinado, funciones, informe final |

---
## Fase 1 — Estructura del proyecto

In [ ]:
import os, shutil
from pathlib import Path

PROYECTO = Path.home() / "ml_auto"
if PROYECTO.exists(): shutil.rmtree(PROYECTO)

# Estructura base
for d in [
    "scripts", "datos/raw", "datos/procesados",
    "experimentos", "logs", "backups", "config"
]:
    (PROYECTO / d).mkdir(parents=True)

os.chdir(PROYECTO)
print(f"Proyecto en: {PROYECTO}")
!ls -R

---
## Fase 2 — Script `setup.sh`

In [ ]:
Path("scripts/setup.sh").write_text("""\
#!/usr/bin/env bash
# setup.sh — prepara el entorno del proyecto ML
set -euo pipefail

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"

log() { echo "[SETUP] $*"; }

log "Inicializando proyecto en: $PROYECTO_DIR"

# Crear directorios necesarios
DIRS=("datos/raw" "datos/procesados" "experimentos" "logs" "backups" "config")
for dir in "${DIRS[@]}"; do
    ruta="$PROYECTO_DIR/$dir"
    if [ ! -d "$ruta" ]; then
        mkdir -p "$ruta"
        log "  Creado: $dir"
    else
        log "  OK:     $dir"
    fi
done

# Hacer ejecutables todos los scripts
chmod +x "$PROYECTO_DIR/scripts"/*.sh
log "Scripts marcados como ejecutables"

# Crear config por defecto si no existe
CONFIG="$PROYECTO_DIR/config/defaults.yaml"
if [ ! -f "$CONFIG" ]; then
    cat > "$CONFIG" << 'EOF'
# Configuración por defecto
epochs: 50
batch_size: 32
learning_rate: 0.001
optimizer: adam
max_backups: 5
log_level: INFO
EOF
    log "Config por defecto creada: $CONFIG"
fi

# Verificar herramientas
log "Verificando herramientas..."
HERRAMIENTAS=(python3 git grep find)
OK=0; FALLOS=0
for tool in "${HERRAMIENTAS[@]}"; do
    if command -v "$tool" &>/dev/null; then
        log "  ✅ $tool"
        ((OK++))
    else
        log "  ❌ $tool — no encontrado"
        ((FALLOS++))
    fi
done

echo ""
log "Setup completado: $OK OK, $FALLOS fallos"
[ $FALLOS -eq 0 ] && exit 0 || exit 1
""", encoding="utf-8")

!chmod +x scripts/setup.sh
print("Ejecutando setup.sh:")
!bash scripts/setup.sh

---
## Fase 3 — Script `lanzar.sh`

In [ ]:
Path("scripts/lanzar.sh").write_text("""\
#!/usr/bin/env bash
# lanzar.sh — lanza un experimento de entrenamiento
# Uso: lanzar.sh <nombre> <lr> [epochs] [batch_size]

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"

NOMBRE=${1:?"Argumento 1 requerido: nombre del experimento"}
LR=${2:?"Argumento 2 requerido: learning rate"}
EPOCHS=${3:-50}
BATCH=${4:-32}

FECHA=$(date +%Y%m%d_%H%M%S)
EXP_DIR="$PROYECTO_DIR/experimentos/${NOMBRE}_${FECHA}"
LOG="$PROYECTO_DIR/logs/${NOMBRE}_${FECHA}.log"

log() { echo "[$(date +%H:%M:%S)] $*" | tee -a "$LOG"; }

# Crear directorio del experimento
mkdir -p "$EXP_DIR"

log "=== EXPERIMENTO: $NOMBRE ==="
log "Fecha:          $FECHA"
log "Learning rate:  $LR"
log "Epochs:         $EPOCHS"
log "Batch size:     $BATCH"

# Guardar configuración
cat > "$EXP_DIR/config.yaml" << EOF
experimento: $NOMBRE
fecha: $FECHA
learning_rate: $LR
epochs: $EPOCHS
batch_size: $BATCH
EOF
log "Config guardada en: $EXP_DIR/config.yaml"

# Simular entrenamiento
log "Iniciando entrenamiento..."
import_err=0
for ((epoch=1; epoch<=EPOCHS; epoch+=10)); do
    LOSS=$(echo "$LR $epoch" | awk '{printf "%.4f", 2.0 * exp(-$2 * 0.05 * $1 * 100)}')
    ACC=$(echo "$LOSS" | awk '{printf "%.4f", 1 - $1/2.5}')
    log "[INFO] Epoch $(printf '%02d' $epoch)/$EPOCHS — loss=$LOSS acc=$ACC"
    if [ $((epoch % 30)) -eq 0 ]; then
        log "[WARNING] Plateau detectado en epoch $epoch"
    fi
done

log "Entrenamiento completado"
log "Log guardado en: $LOG"

echo ""
echo "✅ Experimento $NOMBRE completado"
echo "   Directorio: $EXP_DIR"
echo "   Log:        $LOG"
""", encoding="utf-8")

!chmod +x scripts/lanzar.sh

print("Lanzando experimentos:")
!bash scripts/lanzar.sh modelo_v1 0.001 50 32
print("\n---")
!bash scripts/lanzar.sh modelo_v2 0.01 50 64
print("\n---")
!bash scripts/lanzar.sh modelo_v3 0.0001 50 16

---
## Fase 4 — Script `analizar.sh`

In [ ]:
Path("scripts/analizar.sh").write_text("""\
#!/usr/bin/env bash
# analizar.sh — analiza los logs de entrenamiento

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
LOGS_DIR="$PROYECTO_DIR/logs"
FECHA=$(date +%Y%m%d_%H%M%S)
INFORME="$PROYECTO_DIR/logs/analisis_${FECHA}.txt"

if [ ! -d "$LOGS_DIR" ]; then
    echo "Error: directorio de logs no encontrado" >&2
    exit 1
fi

LOGS=$(find "$LOGS_DIR" -name "*.log" -not -name "analisis_*" -type f | sort)
TOTAL_LOGS=$(echo "$LOGS" | grep -c '.' 2>/dev/null || echo 0)

if [ "$TOTAL_LOGS" -eq 0 ]; then
    echo "No se encontraron logs de entrenamiento"
    exit 0
fi

{
echo "================================================"
echo "  ANÁLISIS DE EXPERIMENTOS — $(date)"
echo "================================================"
echo "Logs encontrados: $TOTAL_LOGS"
echo ""

printf '%-35s %7s %7s %7s\n' "EXPERIMENTO" "EPOCHS" "WARNS" "FINAL_ACC"
printf '%s\n' "$(printf '─%.0s' {1..60})"

MEJOR_EXP=""
MEJOR_ACC=0

while IFS= read -r log; do
    [ -z "$log" ] && continue
    nombre=$(basename "$log" .log)
    epochs=$(grep -c '\[INFO\]' "$log" 2>/dev/null || echo 0)
    warnings=$(grep -c '\[WARNING\]' "$log" 2>/dev/null || echo 0)
    ultima_acc=$(grep -oE 'acc=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    ultima_acc=${ultima_acc:-"0.0000"}

    printf '%-35s %7s %7s %7s\n' "$nombre" "$epochs" "$warnings" "$ultima_acc"

    # Seguimiento del mejor
    if awk "BEGIN {exit !($ultima_acc > $MEJOR_ACC)}" 2>/dev/null; then
        MEJOR_ACC=$ultima_acc
        MEJOR_EXP=$nombre
    fi
done <<< "$LOGS"

echo ""
echo "MEJOR EXPERIMENTO:"
echo "  $MEJOR_EXP (acc=$MEJOR_ACC)"

echo ""
echo "ADVERTENCIAS TOTALES:"
cat $LOGS_DIR/*.log 2>/dev/null | grep -c '\[WARNING\]' | xargs echo " "

echo ""
echo "ERRORES ENCONTRADOS:"
ERRORES=$(cat $LOGS_DIR/*.log 2>/dev/null | grep '\[ERROR\]' || echo "  (ninguno)")
echo "$ERRORES"

} | tee "$INFORME"

echo ""
echo "Informe guardado: $INFORME"
""", encoding="utf-8")

!chmod +x scripts/analizar.sh
print("Ejecutando análisis:")
!bash scripts/analizar.sh

---
## Fase 5 — Script `backup.sh`

In [ ]:
Path("scripts/backup.sh").write_text("""\
#!/usr/bin/env bash
# backup.sh — backup con rotación automática

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
BACKUPS_DIR="$PROYECTO_DIR/backups"
MAX_BACKUPS=${1:-5}
FECHA=$(date +%Y%m%d_%H%M%S)
NOMBRE="backup_${FECHA}.tar.gz"
LOG_BACKUP="$BACKUPS_DIR/backup.log"

log() { echo "[$(date +%H:%M:%S)] $*" | tee -a "$LOG_BACKUP"; }

log "=== BACKUP $FECHA ==="
log "Proyecto: $PROYECTO_DIR"
log "Destino:  $BACKUPS_DIR/$NOMBRE"

# Crear backup excluyendo backups previos y caches
tar -czf "$BACKUPS_DIR/$NOMBRE" \
    --exclude="$PROYECTO_DIR/backups" \
    --exclude="*/__pycache__" \
    --exclude="*.pyc" \
    -C "$(dirname "$PROYECTO_DIR")" \
    "$(basename "$PROYECTO_DIR")" 2>/dev/null

if [ -s "$BACKUPS_DIR/$NOMBRE" ]; then
    TAMANO=$(du -sh "$BACKUPS_DIR/$NOMBRE" | cut -f1)
    log "Backup creado: $NOMBRE ($TAMANO)"
else
    log "ERROR: fallo al crear backup"
    exit 1
fi

# Rotación
TOTAL=$(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)
log "Backups existentes: $TOTAL (máximo: $MAX_BACKUPS)"
ELIMINADOS=0

while [ "$TOTAL" -gt "$MAX_BACKUPS" ]; do
    ANTIGUO=$(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | sort | head -1)
    log "Eliminando antiguo: $(basename "$ANTIGUO")"
    rm "$ANTIGUO"
    ((TOTAL--))
    ((ELIMINADOS++))
done

log "Eliminados: $ELIMINADOS | Actuales: $(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)"
log "=== FIN BACKUP OK ==="

echo ""
echo "✅ Backup: $NOMBRE ($TAMANO)"
""", encoding="utf-8")

!chmod +x scripts/backup.sh
print("Ejecutando backup:")
!bash scripts/backup.sh 3

print("\nContenido de backups/:")
!ls -lh backups/

---
## Fase 6 — Script `estado.sh` (el integrador)

In [ ]:
Path("scripts/estado.sh").write_text("""\
#!/usr/bin/env bash
# estado.sh — informe completo del estado del proyecto

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
NOMBRE_PROYECTO=$(basename "$PROYECTO_DIR")

separador() { printf '%s\n' "$(printf '═%.0s' {1..55})"; }
seccion()    { echo ""; separador; echo "  $1"; separador; }

separador
echo "  ESTADO DEL PROYECTO: $NOMBRE_PROYECTO"
echo "  Fecha: $(date '+%Y-%m-%d %H:%M:%S')"
separador

# ── Estructura ─────────────────────────────────────────────
seccion "ESTRUCTURA"
TOTAL_ARCH=$(find "$PROYECTO_DIR" -type f | wc -l)
TOTAL_DIR=$(find "$PROYECTO_DIR" -type d | wc -l)
TAMANO=$(du -sh "$PROYECTO_DIR" 2>/dev/null | cut -f1)
echo "  Archivos:     $TOTAL_ARCH"
echo "  Directorios:  $TOTAL_DIR"
echo "  Tamaño total: $TAMANO"

# ── Scripts ────────────────────────────────────────────────
seccion "SCRIPTS (.sh)"
for script in "$PROYECTO_DIR/scripts"/*.sh; do
    [ -f "$script" ] || continue
    nombre=$(basename "$script")
    lineas=$(wc -l < "$script")
    ejecutable=$([ -x "$script" ] && echo "✅" || echo "❌")
    printf '  %s %-25s %3d líneas\n' "$ejecutable" "$nombre" "$lineas"
done

# ── Experimentos ───────────────────────────────────────────
seccion "EXPERIMENTOS"
EXP_COUNT=$(find "$PROYECTO_DIR/experimentos" -maxdepth 1 -mindepth 1 -type d | wc -l)
echo "  Total: $EXP_COUNT"
if [ "$EXP_COUNT" -gt 0 ]; then
    echo "  Últimos 3:"
    find "$PROYECTO_DIR/experimentos" -maxdepth 1 -mindepth 1 -type d | \
        sort -r | head -3 | while read -r exp; do
        echo "    $(basename "$exp")"
    done
fi

# ── Logs ───────────────────────────────────────────────────
seccion "LOGS"
LOG_COUNT=$(find "$PROYECTO_DIR/logs" -name "*.log" -not -name "analisis_*" | wc -l)
echo "  Logs de entrenamiento: $LOG_COUNT"
if [ "$LOG_COUNT" -gt 0 ]; then
    TOTAL_ERRORES=$(cat "$PROYECTO_DIR"/logs/*.log 2>/dev/null | grep -c '\[ERROR\]' || echo 0)
    TOTAL_WARNINGS=$(cat "$PROYECTO_DIR"/logs/*.log 2>/dev/null | grep -c '\[WARNING\]' || echo 0)
    echo "  Errores totales:      $TOTAL_ERRORES"
    echo "  Warnings totales:     $TOTAL_WARNINGS"
    if [ "$TOTAL_ERRORES" -gt 0 ]; then
        echo "  ⚠️  Hay errores — ejecuta analizar.sh para detalles"
    fi
fi

# ── Backups ────────────────────────────────────────────────
seccion "BACKUPS"
BACKUP_COUNT=$(find "$PROYECTO_DIR/backups" -name 'backup_*.tar.gz' | wc -l)
echo "  Backups disponibles: $BACKUP_COUNT"
if [ "$BACKUP_COUNT" -gt 0 ]; then
    echo "  Más reciente:"
    find "$PROYECTO_DIR/backups" -name 'backup_*.tar.gz' | sort -r | head -1 | \
        xargs -I{} sh -c 'echo "    $(basename {}) — $(du -sh {} | cut -f1)"'
fi

# ── Config ─────────────────────────────────────────────────
seccion "CONFIGURACIÓN"
CONFIG="$PROYECTO_DIR/config/defaults.yaml"
if [ -f "$CONFIG" ]; then
    echo "  defaults.yaml:"
    grep -v '^#' "$CONFIG" | grep -v '^$' | sed 's/^/    /'
else
    echo "  ⚠️  No existe defaults.yaml — ejecuta setup.sh"
fi

separador
echo "  Comandos disponibles:"
echo "    bash scripts/setup.sh              # inicializar"
echo "    bash scripts/lanzar.sh <n> <lr>    # nuevo experimento"
echo "    bash scripts/analizar.sh           # analizar logs"
echo "    bash scripts/backup.sh [max]       # hacer backup"
separador
""", encoding="utf-8")

!chmod +x scripts/estado.sh
print("Estado del proyecto:")
!bash scripts/estado.sh

---
## Práctica — Extender el sistema

---

### Ejercicio 1 — Script `comparar.sh`

**Enunciado:** Escribe `scripts/comparar.sh` que compara dos experimentos y determina cuál fue mejor:

```bash
bash scripts/comparar.sh experimento_A experimento_B
```

El script debe:
1. Recibir los nombres de dos experimentos como argumentos.
2. Buscar sus logs en `logs/`.
3. Extraer: loss inicial, loss final, accuracy final, número de warnings.
4. Mostrar una tabla comparativa.
5. Declarar un ganador basado en la accuracy final.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from pathlib import Path
import os
os.chdir(Path.home() / "ml_auto")


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```bash
#!/usr/bin/env bash
# comparar.sh — compara dos experimentos

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
EXP_A=${1:?"Argumento 1: nombre del experimento A"}
EXP_B=${2:?"Argumento 2: nombre del experimento B"}

# Buscar log de un experimento
buscar_log() {
    local nombre="$1"
    find "$PROYECTO_DIR/logs" -name "${nombre}*.log" -not -name "analisis_*" | head -1
}

# Extraer métricas de un log
extraer_metricas() {
    local log="$1"
    local loss_inicial warnings acc_final loss_final

    loss_inicial=$(grep -oE 'loss=[0-9.]+' "$log" | head -1 | cut -d= -f2)
    loss_final=$(grep -oE 'loss=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    acc_final=$(grep -oE 'acc=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    warnings=$(grep -c '\[WARNING\]' "$log" 2>/dev/null || echo 0)

    echo "$loss_inicial $loss_final $acc_final $warnings"
}

LOG_A=$(buscar_log "$EXP_A")
LOG_B=$(buscar_log "$EXP_B")

[ -z "$LOG_A" ] && { echo "Error: no se encontró log para '$EXP_A'" >&2; exit 1; }
[ -z "$LOG_B" ] && { echo "Error: no se encontró log para '$EXP_B'" >&2; exit 1; }

read -r L0A LFA ACCA WA <<< "$(extraer_metricas "$LOG_A")"
read -r L0B LFB ACCB WB <<< "$(extraer_metricas "$LOG_B")"

printf '\n%-20s %10s %10s %12s %8s\n' "EXPERIMENTO" "LOSS_INI" "LOSS_FIN" "ACC_FINAL" "WARNS"
printf '%s\n' "$(printf '─%.0s' {1..62})"
printf '%-20s %10s %10s %12s %8s\n' "$EXP_A" "$L0A" "$LFA" "$ACCA" "$WA"
printf '%-20s %10s %10s %12s %8s\n' "$EXP_B" "$L0B" "$LFB" "$ACCB" "$WB"
echo ""

if awk "BEGIN {exit !(${ACCA:-0} > ${ACCB:-0})}" 2>/dev/null; then
    echo "🏆 Ganador: $EXP_A (acc=$ACCA vs $ACCB)"
elif awk "BEGIN {exit !(${ACCB:-0} > ${ACCA:-0})}" 2>/dev/null; then
    echo "🏆 Ganador: $EXP_B (acc=$ACCB vs $ACCA)"
else
    echo "🤝 Empate técnico"
fi
```
</details>

---
### Ejercicio 2 — Script `limpiar.sh`

**Enunciado:** Escribe `scripts/limpiar.sh` que elimina archivos temporales y experimentos antiguos:

1. Elimina archivos `.pyc`, `__pycache__/` y `.tmp` del proyecto.
2. Lista experimentos con más de N días de antigüedad (parámetro opcional, por defecto 7).
3. Pide confirmación antes de borrar experimentos antiguos.
4. Informa del espacio recuperado.
5. Registra la limpieza en `logs/limpieza.log`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Ejercicio 3 — `cron` y automatización periódica

**Enunciado (conceptual + práctica):**

`cron` es el programador de tareas de Linux. Te permite ejecutar scripts automáticamente a una hora o intervalo determinado.

```bash
# Ver las tareas cron del usuario
crontab -l

# Editar
crontab -e

# Formato: minuto hora dia_mes mes dia_semana comando
#  *   *   *   *   *   comando
#  │   │   │   │   └── día semana (0=dom, 7=dom)
#  │   │   │   └────── mes (1-12)
#  │   │   └────────── día del mes (1-31)
#  │   └────────────── hora (0-23)
#  └────────────────── minuto (0-59)
```

**Ejemplos:**
```bash
0 2 * * *  /home/user/ml_auto/scripts/backup.sh        # backup cada día a las 2:00
*/15 * * * * /home/user/ml_auto/scripts/analizar.sh    # análisis cada 15 minutos
0 8 * * 1  /home/user/ml_auto/scripts/estado.sh        # informe cada lunes a las 8:00
```

**Tarea:**
1. Escribe un script `instalar_cron.sh` que añada una entrada cron para hacer backup diario a las 3:00 AM.
2. Verifica con `crontab -l` que se añadió correctamente.
3. Escribe el comando para eliminar esa entrada específica sin borrar otras.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — instalar_cron.sh


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```bash
#!/usr/bin/env bash
# instalar_cron.sh — añade tarea cron de backup

SCRIPT="$HOME/ml_auto/scripts/backup.sh"
LOG="$HOME/ml_auto/logs/cron_backup.log"
ENTRADA="0 3 * * * $SCRIPT >> $LOG 2>&1"

# Verificar que el script existe
if [ ! -f "$SCRIPT" ]; then
    echo "Error: $SCRIPT no existe" >&2
    exit 1
fi

# Añadir solo si no existe ya
if crontab -l 2>/dev/null | grep -qF "$SCRIPT"; then
    echo "La entrada ya existe en crontab"
else
    (crontab -l 2>/dev/null; echo "$ENTRADA") | crontab -
    echo "✅ Entrada añadida:"
    echo "   $ENTRADA"
fi

echo ""
echo "Crontab actual:"
crontab -l
```

**Para eliminar esa entrada específica:**
```bash
crontab -l | grep -vF "backup.sh" | crontab -
```

**Nota importante:** siempre usa rutas absolutas en el crontab — cuando cron ejecuta el script, el `$PATH` es diferente al de tu shell interactivo y los programas en rutas relativas no se encuentran.
</details>

---
### Ejercicio 4 — Integración Python + Bash

**Enunciado:** Escribe un script Python `orquestar.py` que use `subprocess` para:

1. Llamar a `lanzar.sh` con tres learning rates distintos en secuencia.
2. Esperar a que cada uno termine antes de lanzar el siguiente.
3. Llamar a `analizar.sh` al final.
4. Leer el informe generado y mostrar por pantalla solo las líneas del resumen.
5. Retornar el nombre del mejor experimento.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import subprocess, re, os
from pathlib import Path

ML_AUTO = Path.home() / "ml_auto"
os.chdir(ML_AUTO)

def run(cmd, **kwargs):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    return r

# 1 y 2. Lanzar experimentos
experimentos = [
    ("grid_lr_001",   "0.001"),
    ("grid_lr_005",   "0.005"),
    ("grid_lr_01",    "0.01"),
]

for nombre, lr in experimentos:
    print(f"\n{'─'*40}")
    print(f"Lanzando: {nombre} (lr={lr})")
    r = run(f"bash scripts/lanzar.sh {nombre} {lr} 30")
    if r.returncode != 0:
        print(f"⚠️  Experimento {nombre} falló")

# 3. Analizar
print(f"\n{'─'*40}")
print("Analizando resultados...")
run("bash scripts/analizar.sh")

# 4. Leer informe
informes = sorted(ML_AUTO.glob("logs/analisis_*.txt"), reverse=True)
if informes:
    contenido = informes[0].read_text(encoding="utf-8")
    print("\nRESUMEN DEL INFORME:")
    for linea in contenido.splitlines():
        if "MEJOR" in linea or "Total" in linea or linea.startswith("="):
            print(f"  {linea}")

    # 5. Extraer mejor experimento
    m = re.search(r'MEJOR EXPERIMENTO:\s*\n\s*(\S+)', contenido)
    if m:
        print(f"\n🏆 Mejor experimento: {m.group(1)}")
        return_val = m.group(1)
    else:
        return_val = None
        print("No se pudo determinar el mejor")
```
</details>

---
### Ejercicio 5 (reto) — Makefile

**Enunciado:** Un `Makefile` es la forma clásica de documentar y encadenar comandos en proyectos de software. Es una alternativa más portable a los scripts de setup.

Crea un `Makefile` en la raíz de `ml_auto/` con los siguientes targets:

```makefile
make setup          # ejecuta setup.sh
make experimento N=nombre LR=0.001   # lanza un experimento
make analizar       # ejecuta analizar.sh
make backup         # ejecuta backup.sh
make estado         # ejecuta estado.sh
make limpiar        # elimina .pyc y __pycache__
make help           # muestra los comandos disponibles
```

Después, ejecuta `make help` y `make estado`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```makefile
# Makefile — Sistema de automatización ML
.PHONY: setup experimento analizar backup estado limpiar help

SCRIPTS = scripts
N      ?= experimento_$(shell date +%H%M%S)
LR     ?= 0.001
EPOCHS ?= 50
MAX_BK ?= 5

help:
	@echo ""
	@echo "Sistema de Automatización ML"
	@echo "============================="
	@echo "  make setup                   Inicializar el proyecto"
	@echo "  make experimento N=x LR=0.01 Lanzar experimento"
	@echo "  make analizar                Analizar logs"
	@echo "  make backup                  Hacer backup"
	@echo "  make estado                  Ver estado del proyecto"
	@echo "  make limpiar                 Eliminar archivos temporales"
	@echo ""

setup:
	bash $(SCRIPTS)/setup.sh

experimento:
	bash $(SCRIPTS)/lanzar.sh $(N) $(LR) $(EPOCHS)

analizar:
	bash $(SCRIPTS)/analizar.sh

backup:
	bash $(SCRIPTS)/backup.sh $(MAX_BK)

estado:
	bash $(SCRIPTS)/estado.sh

limpiar:
	find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
	find . -name '*.pyc' -delete 2>/dev/null; true
	find . -name '*.tmp' -delete 2>/dev/null; true
	@echo "Limpieza completada"
```

```python
# Crear y probar desde Python
from pathlib import Path
import os

MAKEFILE = Path.home() / "ml_auto" / "Makefile"
# (escribe el contenido arriba)

os.chdir(Path.home() / "ml_auto")
!make help
!make estado
```

**`?=`** define una variable con valor por defecto solo si no fue definida. **`.PHONY`** indica que esos targets no generan archivos — sin él, si existiera un archivo llamado `setup`, `make setup` no haría nada.
</details>

---
## Reflexión final — ¿Qué hemos construido?

El sistema `ml_auto/` es una infraestructura real. Con cinco scripts has automatizado:

| Tarea | Sin automatización | Con automatización |
|-------|------------------|-----------------|
| Preparar entorno | Crear cada carpeta a mano | `bash setup.sh` |
| Lanzar experimentos | Teclear parámetros cada vez | `bash lanzar.sh nombre 0.001` |
| Analizar resultados | Leer logs uno a uno | `bash analizar.sh` |
| Hacer backup | Comprimir y mover a mano | `bash backup.sh` (o cron) |
| Ver el estado | Explorar directorios | `bash estado.sh` |

> Automatizar es una inversión: cuesta tiempo escribir el script, pero lo recuperas cada vez que lo ejecutas.

---
## Módulo 3 completado — Resumen de lo aprendido

```
CLASE 1  Sistema de archivos y navegación
         ├── Estructura del filesystem Linux (/bin, /etc, /home…)
         ├── Rutas absolutas y relativas (., .., ~, -)
         ├── pwd, ls -la, cd, mkdir -p, rmdir
         └── tree y exploración de proyectos

CLASE 2  Gestión de archivos y permisos
         ├── cp, mv, rm (con y sin -r)
         ├── cat, head, tail, less, wc
         ├── touch, nano
         └── Permisos rwx, chmod 755/644/600, chown

CLASE 3  Búsqueda y procesamiento de texto
         ├── grep: -i, -n, -r, -v, -c, -E, -C
         ├── find: -name, -type, -mtime, -size, -exec, -delete
         ├── Pipes |: encadenar comandos
         ├── Redirecciones: >, >>, 2>, 2>&1, /dev/null
         └── wc, sort, uniq, cut

CLASE 4  Bash scripting
         ├── Shebang, set -euo pipefail, chmod +x
         ├── Variables, sustitución $(), aritmética $(())
         ├── Argumentos $1 $2 $@ $# shift
         ├── if/elif/else, operadores -f -d -z -eq -gt
         ├── for (listas, glob, seq) y while
         └── Funciones con local y return

CLASE 5  Proyecto: sistema de automatización ML
         ├── setup.sh, lanzar.sh, analizar.sh
         ├── backup.sh con rotación automática
         ├── estado.sh integrador
         ├── cron para tareas periódicas
         └── Makefile como interfaz unificada
```

**Próximo módulo:** Python para datos — NumPy, Pandas y visualización.